In [1]:
import cv2
import numpy as np
import time

# Load grayscale image
img = cv2.imread("image1.jpeg", 0)

if img is None:
    raise ValueError("Image not found. Check file path.")

# -------------------------------
# Padding Function
# -------------------------------
def apply_padding(image, pad_h, pad_w, mode):
    
    if mode == "zero":
        padded = np.zeros((image.shape[0] + 2*pad_h,
                           image.shape[1] + 2*pad_w))
        padded[pad_h:-pad_h, pad_w:-pad_w] = image
        
    elif mode == "replicate":
        padded = np.pad(image,
                        ((pad_h, pad_h), (pad_w, pad_w)),
                        mode='edge')
        
    elif mode == "none":
        return image
    
    else:
        raise ValueError("Invalid padding mode.")
        
    return padded

# -------------------------------
# 2D Convolution from Scratch
# -------------------------------
def convolution2D(image, kernel, padding="zero"):
    
    k_h, k_w = kernel.shape
    pad_h = k_h // 2
    pad_w = k_w // 2
    
    # Flip kernel (true convolution)
    kernel = np.flipud(np.fliplr(kernel))
    
    if padding != "none":
        image_padded = apply_padding(image, pad_h, pad_w, padding)
        output = np.zeros_like(image)
        
        for i in range(image.shape[0]):
            for j in range(image.shape[1]):
                region = image_padded[i:i+k_h, j:j+k_w]
                output[i, j] = np.sum(region * kernel)
    else:
        output_h = image.shape[0] - k_h + 1
        output_w = image.shape[1] - k_w + 1
        output = np.zeros((output_h, output_w))
        
        for i in range(output_h):
            for j in range(output_w):
                region = image[i:i+k_h, j:j+k_w]
                output[i, j] = np.sum(region * kernel)
    
    return output


In [2]:
# 3x3 Box Filter
box_filter = (1/9) * np.ones((3,3))

# Identity Kernel
identity_kernel = np.array([[0,0,0],
                            [0,1,0],
                            [0,0,0]])

# Edge Detection Kernel
edge_kernel = np.array([[-1,-1,-1],
                        [-1, 8,-1],
                        [-1,-1,-1]])


In [3]:
kernels = {
    "Box Filter": box_filter,
    "Identity": identity_kernel,
    "Edge Detection": edge_kernel
}

padding_modes = ["zero", "replicate", "none"]

for name, kernel in kernels.items():
    print(f"\nKernel: {name}")
    
    for pad in padding_modes:
        
        start = time.time()
        custom_output = convolution2D(img, kernel, padding=pad)
        end = time.time()
        
        custom_time = end - start
        
        # OpenCV comparison (only for padded cases)
        if pad != "none":
            start_cv = time.time()
            cv_output = cv2.filter2D(img, -1, kernel)
            end_cv = time.time()
            cv_time = end_cv - start_cv
            
            difference = np.mean(np.abs(custom_output - cv_output))
            
            print(f"Padding: {pad}")
            print(f"Custom Time: {custom_time:.5f} sec")
            print(f"OpenCV Time: {cv_time:.5f} sec")
            print(f"Mean Difference: {difference:.5f}")
            
        else:
            print(f"Padding: none")
            print(f"Custom Time: {custom_time:.5f} sec")



Kernel: Box Filter
Padding: zero
Custom Time: 0.40531 sec
OpenCV Time: 0.01165 sec
Mean Difference: 121.27315
Padding: replicate
Custom Time: 0.45545 sec
OpenCV Time: 0.00014 sec
Mean Difference: 118.50137
Padding: none
Custom Time: 0.39040 sec

Kernel: Identity
Padding: zero
Custom Time: 0.42841 sec
OpenCV Time: 0.00039 sec
Mean Difference: 0.00000
Padding: replicate
Custom Time: 0.50725 sec
OpenCV Time: 0.00009 sec
Mean Difference: 0.00000
Padding: none
Custom Time: 0.51440 sec

Kernel: Edge Detection
Padding: zero
Custom Time: 0.89279 sec
OpenCV Time: 0.00021 sec
Mean Difference: 106.11185
Padding: replicate
Custom Time: 0.86557 sec
OpenCV Time: 0.00020 sec
Mean Difference: 107.95019
Padding: none
Custom Time: 0.59368 sec
